# Solution C1 · Too many rounds: the accuracy stays, the probabilities get ruined

**Type**: Break the model

## Objective

Train XGBoost with a high learning rate and deep trees for 400 rounds and observe, on the test set, **accuracy, log-loss and calibration** as a function of the rounds: the log-loss touches its minimum around 25 rounds and then rises by 45%, while the accuracy stays almost still. The training log-loss goes to almost zero.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

X, y = make_classification(6000, n_features=12, n_informative=5, class_sep=0.7, flip_y=0.1, random_state=0)
X_tr, X_rest, y_tr, y_rest = train_test_split(X, y, test_size=0.5, random_state=0)
X_val, X_te, y_val, y_te = train_test_split(X_rest, y_rest, test_size=0.6, random_state=0)
m = xgb.XGBClassifier(n_estimators=400, learning_rate=0.3, max_depth=6, eval_metric="logloss", n_jobs=-1, random_state=0).fit(X_tr, y_tr)
ks = [10, 25, 50, 100, 200, 400]

## Commented solution

In [ ]:
table = pd.DataFrame(index=ks, columns=["train_ll", "test_ll", "test_acc"], dtype=float)
for k in ks:
    p_tr = m.predict_proba(X_tr, iteration_range=(0, k))[:, 1]
    p_te = m.predict_proba(X_te, iteration_range=(0, k))[:, 1]
    table.loc[k] = [log_loss(y_tr, p_tr), log_loss(y_te, p_te), ((p_te > 0.5) == y_te).mean()]
table.round(3)

In [ ]:
k_min_ll = int(table["test_ll"].idxmin())
min_ll, ll_400 = float(table.loc[k_min_ll, "test_ll"]), float(table.loc[400, "test_ll"])
ll_increase = ll_400 / min_ll - 1
acc_change = float(table["test_acc"].max() - table.loc[400, "test_acc"])
print(f"test log-loss: minimum {min_ll:.3f} at {k_min_ll} rounds, at 400 rounds {ll_400:.3f} ({ll_increase:+.0%}) | accuracy: maximum change {acc_change:.3f}")

In [ ]:
def share(k):
    p = m.predict_proba(X_te, iteration_range=(0, k))[:, 1]
    extreme = (p < 0.02) | (p > 0.98)
    return float(extreme.mean()), float((((p > 0.5) != y_te) & extreme).sum() / max(extreme.sum(), 1))


extreme_share = {k: share(k)[0] for k in (25, 400)}
extreme_err = share(400)[1]
print(f"extreme probabilities (below 0.02 or above 0.98): {extreme_share} | at 400 rounds, among the extreme ones {extreme_err:.0%} get the class wrong")

In [ ]:
m_es = xgb.XGBClassifier(n_estimators=400, learning_rate=0.3, max_depth=6, eval_metric="logloss", early_stopping_rounds=30, n_jobs=-1, random_state=0)
m_es.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
early_round = m_es.best_iteration + 1
early_test_ll = log_loss(y_te, m_es.predict_proba(X_te)[:, 1])
print(f"early stopping: {early_round} rounds | test log-loss {early_test_ll:.3f} (at 400 rounds: {ll_400:.3f})")

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and table["train_ll"].is_monotonic_decreasing and table.loc[400, "train_ll"] < 0.02, "the training log-loss always drops, down to under 0.02", "iteration_range=(0, k)")
    check(k_min_ll in (10, 25, 50) and ll_increase > 0.35 and ll_400 > 0.65, "the test log-loss has its minimum between 10 and 50 rounds and at 400 it is over 35% higher (over 0.65)", "compare the rows of the table")
    check(abs(acc_change) < 0.02 and table["test_acc"].min() > 0.78, "the test accuracy varies by less than 2 points: the problem is not visible with predict", "maximum test_acc minus test_acc at 400")
    check(extreme_share[400] > 0.4 and extreme_share[25] < 0.1 and extreme_err > 0.05, "at 400 rounds over 40% of the probabilities are extreme (below 0.02 or above 0.98), at 25 less than 10%; among the extreme ones over 5% are wrong", "(p < 0.02) | (p > 0.98)")
    check(early_round < 100 and early_test_ll < 0.5 and early_test_ll < ll_400 - 0.15, "early stopping halts before 100 rounds and gives a log-loss below 0.5", "early_stopping_rounds=30 on X_val")
done()

## What you learned

- **The training log-loss goes to zero, the test one rises by 45%:** from 0.473 at 25 rounds to 0.683 at 400. The model is more and more sure of itself (at 400 rounds 52% of the test probabilities are below 0.02 or above 0.98, at 25 rounds 1.5%) and it is wrong with more and more confidence: among the extreme probabilities 12% are on the wrong side. Every wrong prediction with probability 0.99 costs the log-loss a great deal.
- **The accuracy does not notice:** it stays at 0.79-0.80 all the way: the class is decided with the 0.5 threshold and changes only when a probability crosses 0.5, while the overfitting makes the probabilities more extreme without moving them to the other side. Anyone who looks only at `score()` may never notice the problem.
- **It matters if you use the probabilities:** for an expected cost (M4 B2), a ranking of customers or a calibration, those extreme probabilities are wrong. That is why early stopping is done on the **log-loss** (or on a probabilistic metric) and not on the classification error.
- **Early stopping fixes it:** 30 rounds of patience after the minimum stop the model at 15 rounds with a test log-loss of 0.466: like the absolute minimum of the table (0.473) and without looking at the test set.

## Common pitfall

Evaluating a boosting model with `score()` (accuracy) after increasing the rounds. Accuracy is the last metric to notice the overfitting: look at the log-loss and the calibration.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why is early stopping done on a probabilistic metric like the log-loss?</summary>

Because the overfitting of boosting shows up first in the probabilities (which become extreme and wrong) than in the thresholded classification: the accuracy can stay almost constant while the validation log-loss is already rising a lot. A probabilistic metric picks up the deterioration right away; if the business metric is another
(AUC, costs), you monitor it together.

</details>